# Task: Determine the negative-filtering threshold empirically

## Context

I am building a daily flood occurrence classifier for the Mount Elgon region of
Eastern Uganda (districts: Mbale, Bududa, Sironko, Manafwa, Butaleja,
Kapchorwa, Bulambuli, and optionally Kween and Bukwo). This replicates
Sankaranarayanan et al. (2020), "Flood prediction based on weather parameters
using deep learning", but at daily rather than monthly resolution.

Positive days are extremely rare, roughly 280 flood records against ~80,000
district-days, giving a prevalence near 0.2%. Before training, I need to
restrict the negative class to days where flooding was physically plausible,
so the model learns the boundary between "heavy rain that flooded" and "heavy
rain that did not", rather than the trivial rule that most days are dry.

Your job is to determine that threshold empirically from my data. Do not train
any model. This is a data analysis task that produces a justified number and
the evidence behind it.

## Inputs

- Labelled flood occurrence data: `dataset/flood_labelled_data.csv`
  Expected columns (adapt to what is actually there): `Serial`,`Date (YMD)`,`Day Index`,`Observation Date`,`District`,`Duration (d)`,`Event Days`,`Duration Class`.
  Each row is one recorded flood event for one district.

- Daily rainfall data: `dataset/chirps_daily_rainfall.csv`
  Expected columns: `date,district`,`rain_mean`,`rain_max`,`rain_min`.
  This is catchment-mean daily rainfall in millimetres, extracted per district.

- Output directory for tables and figures: `dataset/`

Inspect both files before assuming anything about their structure, dtypes,
date formats or units. Report what you actually find, including row counts,
date ranges, districts present, and any missing or duplicated records.

## Definitions

- A **positive day** is the onset date of a recorded flood event for that
  district. Use the event start date only. Do not expand events across multiple
  days, even where a duration field exists.
- A **negative day** is any other district-day in the record.
- **Antecedent rainfall** is the rolling cumulative catchment rainfall over a
  window of N days ending on and including the day in question. Compute this
  per district, never pooled across districts, and make sure the rolling window
  does not run across gaps in the date index.

## Steps

1. **Load and join.** Build a complete daily panel of district-days across the
   full overlapping date range of both sources. Mark each day 1 or 0. Report
   the resulting positive count, total row count and prevalence. Flag any flood
   records that could not be matched to rainfall data, and any dates that look
   month-precise rather than day-precise (for example a day value of 1 or 0
   appearing far more often than chance). Exclude unmatched or
   month-precise records from the threshold analysis and report how many.

2. **Compute antecedent rainfall** for windows of 1, 2, 3, 5, 7 and 15 days.

3. **Describe the positive distribution.** For each window, report the minimum,
   1st, 5th, 10th, 25th, 50th and 75th percentiles of antecedent rainfall on
   positive days. Do the same for negative days for comparison.

4. **Choose the discriminating window.** For each window, quantify how well
   antecedent rainfall separates positives from negatives, using AUC of the
   single variable and the ratio of positive median to negative median. Report
   which window separates best. Do not assume 3 days is the answer.

5. **Recommend a threshold.** The rule is: retain at least 99% of positives
   while removing as many negatives as possible. State the recommended value,
   how many positives it excludes, and what prevalence and class ratio result.
   Aim for a class ratio that is trainable, roughly in the range 30:1 to 100:1;
   if no threshold achieves both, say so and explain the trade.

6. **Investigate excluded positives individually.** Any flood day falling below
   the recommended threshold is important. List each one with its date,
   district and antecedent rainfall. These are either label errors, month-precise
   dates, non-rainfall triggers such as blocked drainage or upstream release, or
   cases where the gridded rainfall product missed a local convective storm.
   Do not delete them. Report them for manual review.

## Constraints

- Never filter positive days. The threshold applies only to negatives.
- The rule must depend only on rainfall, which is available at prediction time.
  Do not use any variable derived from the outcome.
- Compute rolling windows within district groups, sorted by date.
- Handle missing rainfall days explicitly. State whether you dropped them,
  treated them as zero, or interpolated, and justify the choice.
- Use whatever units are actually in the file. If rainfall is not in mm,
  convert and say so.

## Outputs

Write to `dataset/`:

1. `threshold_analysis.md` — findings, the recommended threshold with its
   justification, the sensitivity comparison, and a short section listing the
   excluded positives for manual review.
2. A plot of antecedent rainfall distributions for positives against negatives
   at the chosen window, with the recommended threshold marked.
3. The analysis script itself, so the work is reproducible.

Report the numbers you find rather than the numbers you expect. If the data
does not support a clean threshold, say so plainly and explain why.